# 03 - Encode Symile ECG waveforms with HuBERT-ECG

Each 10-second, 12-lead waveform is z-normalized per lead, serialized in standard lead order, encoded by the frozen HuBERT-ECG model, and mean-pooled. No ECG label or generated report is used.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import AutoModel

ARTIFACT_ROOT = Path(os.environ.get(
    "ECG_CXR_BRIDGE_ARTIFACT_ROOT",
    "/path/to/ecg_cxr_bridge_artifacts",
))
config = json.loads((ARTIFACT_ROOT / "config.json").read_text())
SYMILE_ROOT = Path(config["symile_root"])
FEATURE_ROOT = ARTIFACT_ROOT / "features"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32
BATCH_SIZE = int(os.environ.get("HUBERT_BATCH_SIZE", "8"))
SPLITS = os.environ.get("BRIDGE_FEATURE_SPLITS", ",".join(config["splits"])).split(",")
print({"device": str(DEVICE), "batch_size": BATCH_SIZE, "splits": SPLITS})


In [ ]:
encoder = AutoModel.from_pretrained(
    config["hubert_model_id"], cache_dir=config["hf_cache"],
    trust_remote_code=True, torch_dtype=DTYPE,
).to(DEVICE).eval().requires_grad_(False)
print("Loaded:", config["hubert_model_id"])


In [ ]:
def hubert_input(batch):
    signal = np.asarray(batch, dtype="float32")
    if signal.ndim == 4:
        signal = signal[:, 0]
    if signal.ndim != 3 or signal.shape[1:] != (5000, 12):
        raise ValueError(f"Expected [B,1,5000,12] or [B,5000,12], got {signal.shape}")
    mean = np.nanmean(signal, axis=1, keepdims=True)
    std = np.nanstd(signal, axis=1, keepdims=True)
    std[std < 1e-6] = 1.0
    signal = np.nan_to_num((signal - mean) / std).astype("float32")
    return signal.transpose(0, 2, 1).reshape(len(signal), -1)

def pooled(output):
    hidden = output.last_hidden_state if hasattr(output, "last_hidden_state") else output[0]
    return hidden.mean(1) if hidden.ndim == 3 else hidden

@torch.inference_mode()
def encode_batch(batch):
    values = torch.from_numpy(hubert_input(batch)).to(DEVICE, dtype=DTYPE)
    return pooled(encoder(input_values=values)).float().cpu().numpy()


In [ ]:
def encode_split(split):
    split_dir = FEATURE_ROOT / split
    split_dir.mkdir(parents=True, exist_ok=True)
    ecg = np.load(SYMILE_ROOT / "data_npy" / split / f"ecg_{split}.npy", mmap_mode="r")
    hadm = np.load(SYMILE_ROOT / "data_npy" / split / f"hadm_id_{split}.npy", mmap_mode="r")
    unique_hadm, first_index, inverse = np.unique(np.asarray(hadm), return_index=True, return_inverse=True)
    probe = encode_batch(ecg[first_index[:1]])
    unique = np.lib.format.open_memmap(split_dir / "ecg_hubert_unique.npy", mode="w+",
                                        dtype="float32", shape=(len(unique_hadm), probe.shape[1]))
    unique[:1] = probe
    for start in tqdm(range(1, len(unique_hadm), BATCH_SIZE), desc=f"HuBERT {split}"):
        stop = min(start + BATCH_SIZE, len(unique_hadm))
        unique[start:stop] = encode_batch(ecg[first_index[start:stop]])
        unique.flush()
    output = np.lib.format.open_memmap(split_dir / "ecg_hubert.npy", mode="w+",
                                        dtype="float32", shape=(len(ecg), probe.shape[1]))
    for start in range(0, len(ecg), 4096):
        stop = min(start + 4096, len(ecg))
        output[start:stop] = unique[inverse[start:stop]]
    output.flush()
    metadata = {"split": split, "rows": len(ecg), "unique_admissions": len(unique_hadm),
                "embedding_dim": probe.shape[1], "model_id": config["hubert_model_id"],
                "pooling": "mean_last_hidden_state", "complete": True}
    (split_dir / "ecg_hubert_metadata.json").write_text(json.dumps(metadata, indent=2))
    print(split, output.shape)

for split in SPLITS:
    encode_split(split)


In [ ]:
for split in SPLITS:
    array = np.load(FEATURE_ROOT / split / "ecg_hubert.npy", mmap_mode="r")
    assert all(np.isfinite(array[start:start + 4096]).all() for start in range(0, len(array), 4096))
print("All ECG feature files are finite.")
